In [ ]:
# =============================================================================
# ADVANCED RAG — give the model a LIBRARY, not just a memory
# =============================================================================
#
# Course map (Phase 11):
#   RAG basics → Chunking & Reranking → Embeddings → Query rewriting → Eval (RAGAS)
#
# Deliverable today:
#   A modular rag_engine that ingests docs, chunks, hybrid-retrieves, reranks,
#   rewrites queries, evaluates, and can plug in as an assistant TOOL.
#
# ---------------------------------------------------------------------------
# THE PROBLEM RAG SOLVES (easy picture)
# ---------------------------------------------------------------------------
# An LLM alone is a smart student who took a big exam months ago (pretrain)
# and maybe a short cram session (SFT). It does NOT magically know YOUR
# latest sales playbook / CRM notes / PDF pricing sheet.
#
# Without RAG:  "What's Enterprise support SLA?" → may GUESS / hallucinate
# With RAG:     look up the playbook page → THEN answer from that text
#
# Sticky analogy:
#   LLM  = brain that can write sentences
#   RAG  = open-book exam (retrieve pages, then generate)
#
# ---------------------------------------------------------------------------
# BASIC RAG LOOP (always start here)
# ---------------------------------------------------------------------------
#   1) INDEX TIME (once / when docs change)
#        docs → chunk → embed each chunk → store in a vector index (+ maybe BM25)
#
#   2) QUERY TIME (every user question)
#        question → (optional rewrite) → retrieve top-k chunks
#                → (optional rerank) → stuff into prompt → LLM answers
#
#   ┌─────────┐    ┌──────────┐    ┌──────────┐    ┌─────┐
#   │ Question│ →  │ Retrieve │ →  │  Prompt  │ →  │ LLM │ → Answer
#   └─────────┘    │  chunks  │    │ + context│    └─────┘
#                  └────▲─────┘    └──────────┘
#                       │
#                  ┌────┴─────┐
#                  │  Index   │  (your PDFs / playbook pieces)
#                  └──────────┘
#
# For your earpiece product:
#   User said something about pricing → retrieve Pricing/Security sections
#   → Correction-GPT answers with grounded facts (less guessing).
#
# ---------------------------------------------------------------------------
# ABBREVIATIONS
# ---------------------------------------------------------------------------
#   RAG   = Retrieval-Augmented Generation
#           Fetch relevant text first, then generate the answer from it.
#
#   BM25  = Best Matching 25 (classic keyword search score)
#           Good at exact terms like "SOC 2", "$299", "SLA".
#
#   Dense / embedding retrieval
#           Meaning search: "how much for pro plan?" ≈ "Professional is $299…"
#
#   Hybrid = BM25 + dense together
#           Keywords AND meaning — usually stronger than either alone.
#
#   Cross-encoder reranker
#           Second-pass judge: score (query, chunk) pairs more carefully.
#
# ---------------------------------------------------------------------------
# ADVANCED PIECES (what "Advanced RAG" adds)
# ---------------------------------------------------------------------------
#
# 1) CHUNKING — how you slice documents
# 2) EMBEDDINGS — turn text into number-arrows
# 3) HYBRID INDEX — BM25 + vectors
# 4) RERANKING — refine the shortlist
# 5) QUERY REWRITING — fix the question before search
# 6) EVALUATION — Precision/Recall/MRR + RAGAS-style faithfulness
#
# ---------------------------------------------------------------------------
# HOW THIS PLUGS INTO YOUR ASSISTANT (MCP / tools)
# ---------------------------------------------------------------------------
# RAG engine can be a tool: search_playbook(query) → top chunks as text.
# Function-calling / MCP loop: model requests search → you retrieve →
# model writes the final earpiece line from real playbook text.
#
print("Advanced RAG map loaded — next: playbook data + chunking.")


In [3]:
# Prepare the Sales Playbook Dataset

# For the demo we'll create a multi‑section sales playbook as text, but you can later point it at real PDFs.

# Sample sales playbook (in practice, load from PDFs)
playbook = [
    {
        "title": "Pricing",
        "content": "Our Starter plan is $99/month, Professional is $299/month, and Enterprise is custom-priced. All plans include 24/7 support. Enterprise customers receive a dedicated account manager."
    },
    {
        "title": "Security",
        "content": "We are SOC 2 Type II compliant. Data is encrypted at rest and in transit. Enterprise customers can use their own encryption keys (BYOK). Annual penetration tests are conducted."
    },
    {
        "title": "API Limits",
        "content": "Rate limits are 1000 requests/minute for Starter, 5000 for Professional, and unlimited for Enterprise. Burst limits are double the base rate."
    },
    {
        "title": "Support",
        "content": "Standard support is 24/7 via chat and email. Enterprise customers get phone support with 1-hour response time SLA. Critical issues are escalated immediately."
    }
]

In [7]:
# =============================================================================
# CHUNKING STRATEGIES — how we slice the playbook into index cards
# =============================================================================
#
# Why chunk at all?
#   Retrievers search small pieces, not whole books.
#   Bad cuts → answer facts split across cards or mixed with unrelated text.
#
# Three strategies in this cell:
#
#   1) FIXED-SIZE (by words) + OVERLAP
#      Walk the word list in steps of (chunk_size - overlap).
#      overlap keeps border words on BOTH neighboring chunks so a sentence
#      isn't destroyed at the cut.
#
#        words:  [0........50) [40........90) [80...]
#                 chunk0        chunk1         chunk2
#                      overlapping 10 words ─┘
#
#   2) SENTENCE-AWARE
#      Split on ".!?" then pack up to N sentences per chunk.
#      Cleaner to read; still ignores "topic change" inside a long para.
#
#   3) SEMANTIC (demo version)
#      True semantic chunking uses embeddings to split when meaning shifts.
#      Here we SIMULATE it: each playbook section (Pricing, Security, …)
#      is already one natural topic → one chunk + metadata title.
#
# Sticky: chunking is INDEX-TIME. Do it before building BM25/vector stores.
#

import re
from typing import List, Dict


def fixed_size_chunks(text: str, chunk_size: int = 100, overlap: int = 20) -> List[str]:
    """Slice text into word windows of length chunk_size with overlap."""
    words = text.split()
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        if i + chunk_size >= len(words):
            break  # avoid tiny trailing duplicate from overlap step
    return chunks


def sentence_aware_chunks(text: str, max_chunk_sentences: int = 3) -> List[str]:
    """Group whole sentences; flush every max_chunk_sentences."""
    # Split AFTER punctuation+space, keeping sentence boundaries
    sentences = re.split(r"(?<=[.!?]) +", text)
    chunks = []
    current = []
    for sent in sentences:
        if not sent.strip():
            continue
        current.append(sent)
        if len(current) >= max_chunk_sentences:
            chunks.append(" ".join(current))
            current = []
    if current:
        chunks.append(" ".join(current))
    return chunks


def semantic_chunks(documents: List[Dict]) -> List[Dict]:
    """Demo semantic boundaries = author sections (title + content)."""
    # Real systems: embed sliding windows, split when similarity drops.
    chunks = []
    for doc in documents:
        chunks.append({
            "text": doc["content"],
            "metadata": {"title": doc["title"]},
        })
    return chunks


# ---------------------------------------------------------------------------
# Demo on the playbook (from previous cell)
# ---------------------------------------------------------------------------
print("=== Fixed-size (chunk_size=50 words, overlap=10) ===")
for doc in playbook:
    chunks = fixed_size_chunks(doc["content"], chunk_size=50, overlap=10)
    print(f"{doc['title']}: {len(doc['content'].split())} words → {len(chunks)} chunk(s)")
    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i}: {chunk[:80]}...")

print("\n=== Sentence-aware (max 3 sentences) ===")
for doc in playbook:
    chunks = sentence_aware_chunks(doc["content"], max_chunk_sentences=3)
    print(f"{doc['title']}: {len(chunks)} chunk(s) → {chunks[0][:80]}...")

print("\n=== Semantic (section = one chunk) ===")
sem = semantic_chunks(playbook)
for ch in sem:
    print(f"  [{ch['metadata']['title']}] {ch['text'][:70]}...")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# Fixed-size lines like:
#   Pricing: 24 words → 1 chunk(s)
#   Chunk 0: Our Starter plan is $99/month, Professional is $299/month...
#
#   Why only 1 chunk? These demo sections are SHORT (~20–30 words).
#   chunk_size=50 means "up to 50 words per card" — the whole section fits
#   on one card, so you won't see overlap splitting yet.
#   On a long PDF paragraph (200+ words) you'd see Chunk 0, 1, 2... with
#   shared border words from overlap=10.
#
# Sentence-aware:
#   Usually 1 chunk per section here too (each section has ≤3 sentences).
#   Longer FAQ answers would pack every 3 sentences into a new card.
#
# Semantic demo:
#   [Pricing], [Security], [API Limits], [Support] — one card each, with
#   metadata title kept (useful later for citations: "according to Security…").
#
# Takeaway: for THIS tiny playbook, section-level (semantic demo) is the
# natural unit. Fixed/sentence settings matter more on bigger documents.


=== Fixed-size (chunk_size=50 words, overlap=10) ===
Pricing: 24 words → 1 chunk(s)
  Chunk 0: Our Starter plan is $99/month, Professional is $299/month, and Enterprise is cus...
Security: 29 words → 1 chunk(s)
  Chunk 0: We are SOC 2 Type II compliant. Data is encrypted at rest and in transit. Enterp...
API Limits: 21 words → 1 chunk(s)
  Chunk 0: Rate limits are 1000 requests/minute for Starter, 5000 for Professional, and unl...
Support: 23 words → 1 chunk(s)
  Chunk 0: Standard support is 24/7 via chat and email. Enterprise customers get phone supp...

=== Sentence-aware (max 3 sentences) ===
Pricing: 1 chunk(s) → Our Starter plan is $99/month, Professional is $299/month, and Enterprise is cus...
Security: 2 chunk(s) → We are SOC 2 Type II compliant. Data is encrypted at rest and in transit. Enterp...
API Limits: 1 chunk(s) → Rate limits are 1000 requests/minute for Starter, 5000 for Professional, and unl...
Support: 1 chunk(s) → Standard support is 24/7 via chat and email. Enterpri

In [9]:
# =============================================================================
# BUILD THE VECTOR INDEX (FAISS) — meaning search over playbook chunks
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC / TERM MAP
# ---------------------------------------------------------------------------
#   Embedding
#     A list of numbers (a vector) that represents the *meaning* of a text.
#     Similar sentences → similar vectors (arrows pointing a similar way).
#
#   SentenceTransformer / MiniLM (all-MiniLM-L6-v2)
#     A small model that turns a sentence/paragraph into one embedding vector.
#     Same family of idea as Word2Vec, but for whole passages.
#
#   Vector index
#     A searchable library of those vectors (one per chunk) so we can ask:
#     "which stored chunks are nearest to this question's vector?"
#
#   FAISS (Facebook AI Similarity Search)
#     A fast library for nearest-neighbor search over many vectors.
#
#   IndexFlatL2
#     Exact search using L2 distance (Euclidean): smaller distance = closer.
#     "Flat" = brute-force over all vectors (fine for tiny corpora; huge
#     corpora use approximate indexes).
#
#   top_k
#     How many nearest chunks to return (here 3).
#
#   Metadata
#     Extra labels stored beside each vector (e.g. section title "Pricing")
#     so you can cite where the text came from.
#
# ---------------------------------------------------------------------------
# PICTURE
# ---------------------------------------------------------------------------
#   INDEX TIME:
#     playbook sections → embed_model.encode → vectors → FAISS.add
#
#   QUERY TIME:
#     "price of Professional?" → encode → FAISS.search → nearest chunk texts
#
#   ┌──────────────┐     embed      ┌─────────┐
#   │ Pricing text │ ─────────────► │ vector0 │──┐
#   │ Security …   │ ─────────────► │ vector1 │──┼─► FAISS index
#   │ API Limits…  │ ─────────────► │ vector2 │──┤
#   └──────────────┘                └─────────┘  │
#                                                ▼
#   question ──embed──► q_vec ──search──► closest vectors → chunk text
#

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load a pretrained embedding model (downloads once, then cached)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Corpus = the texts we will search; metadata[i] describes corpus[i]
corpus = []
metadata = []
for doc in playbook:
    corpus.append(doc["content"])                 # one semantic chunk per section
    metadata.append({"title": doc["title"]})

# Encode all chunks → matrix shape (num_chunks, embedding_dim)
corpus_embeddings = embed_model.encode(corpus, convert_to_numpy=True)
dimension = corpus_embeddings.shape[1]            # e.g. 384 for MiniLM-L6

# Build FAISS index and add every chunk vector
index = faiss.IndexFlatL2(dimension)
index.add(corpus_embeddings)                      # now searchable


def retrieve(query: str, top_k: int = 3):
    """Embed the query and return the top_k nearest playbook chunks."""
    query_embedding = embed_model.encode([query])
    # distances: L2 to each hit; indices: row numbers into corpus/metadata
    distances, indices = index.search(query_embedding, top_k)
    results = []
    for idx_i, dist in zip(indices[0], distances[0]):
        if idx_i < 0:
            continue  # FAISS may pad with -1 if index is smaller than top_k
        results.append({
            "score": float(dist),          # lower L2 = closer / more similar
            "text": corpus[idx_i],
            "metadata": metadata[idx_i],
        })
    return results


# Test: meaning search should prefer the Pricing section
results = retrieve("What is the price of Professional?")
for r in results:
    print(r["metadata"]["title"], r["text"][:100])

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# Expect Pricing near the top, e.g.:
#   Pricing Our Starter plan is $99/month, Professional is $299/month...
#   ... then maybe Support / API / Security with weaker relation
#
# Why Pricing wins: the question and Pricing chunk share meaning
# ("Professional", money/price), so their embeddings sit close in vector space.
#
# Next upgrades: hybrid BM25+dense, then cross-encoder rerank, query rewrite.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7721.14it/s]


Pricing Our Starter plan is $99/month, Professional is $299/month, and Enterprise is custom-priced. All plan
API Limits Rate limits are 1000 requests/minute for Starter, 5000 for Professional, and unlimited for Enterpris
Support Standard support is 24/7 via chat and email. Enterprise customers get phone support with 1-hour resp


In [13]:
# =============================================================================
# HYBRID RETRIEVAL — BM25 (keywords) + dense vectors (meaning)
# =============================================================================
#
# Why hybrid?
#   Dense (FAISS/MiniLM): great at paraphrases ("fancy plan cost" ≈ Professional $299)
#   BM25 (keywords):      great at exact tokens ("SOC", "2", "BYOK", "SLA")
#   Together: fewer misses when the user wording ≠ doc wording OR when rare
#   IDs/acronyms must match exactly.
#
# Picture:
#   query ──► dense scores (per chunk) ──┐
#            BM25 scores (per chunk)  ──┼─► weighted mix → sort → top_k
#                                       │
#   alpha: 1.0 = only dense,  0.0 = only BM25,  0.5 = equal blend
#
# Terms:
#   BM25 / BM25Okapi — classic sparse search score over word counts
#   tokenized_corpus — each chunk split into words for BM25
#   normalize        — put both score types on ~0..1 so mixing is fair
#   L2 → similarity  — FAISS gives distance (lower=better); we flip to
#                      similarity (higher=better) before blending
#

from rank_bm25 import BM25Okapi

# BM25 needs a bag-of-words per chunk (same order as `corpus`)
tokenized_corpus = [doc.split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)


def hybrid_retrieve(query: str, top_k: int = 3, alpha: float = 0.5):
    """Blend dense similarity + BM25; return top_k chunks (higher score = better)."""

    # --- Dense: L2 distance for EVERY chunk, kept in CORPUS ORDER ---
    # BUGFIX: do not zip retrieve()'s sorted hits with bm25_norm[i] —
    # retrieve() returns nearest-first, not index-aligned to corpus.
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, len(corpus))
    dense_dist = [0.0] * len(corpus)
    for idx_i, dist in zip(indices[0], distances[0]):
        if idx_i >= 0:
            dense_dist[int(idx_i)] = float(dist)

    max_dist = max(dense_dist) if dense_dist and max(dense_dist) > 0 else 1.0
    dense_sim = [1.0 - (d / max_dist) for d in dense_dist]  # higher = closer

    # --- BM25: one score per corpus row (already corpus-aligned) ---
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)
    max_bm25 = float(max(bm25_scores)) if len(bm25_scores) and max(bm25_scores) > 0 else 1.0
    bm25_norm = [float(s) / max_bm25 for s in bm25_scores]

    # --- Weighted combination (same i = same chunk) ---
    combined = []
    for i in range(len(corpus)):
        combined_score = alpha * dense_sim[i] + (1.0 - alpha) * bm25_norm[i]
        combined.append({
            "score": combined_score,
            "text": corpus[i],
            "metadata": metadata[i],
        })

    combined.sort(key=lambda x: x["score"], reverse=True)
    return combined[:top_k]


# Test: keywords "security" + "compliance" should surface Security (SOC 2)
print("Hybrid results for 'enterprise security compliance':")
for r in hybrid_retrieve("enterprise security compliance", top_k=2):
    print(f"{r['metadata']['title']}: {r['text'][:100]} (score: {r['score']:.2f})")

# Expect Security first (or very high). Pricing/API should rank lower.


Hybrid results for 'enterprise security compliance':
Security: We are SOC 2 Type II compliant. Data is encrypted at rest and in transit. Enterprise customers can u (score: 0.18)
Support: Standard support is 24/7 via chat and email. Enterprise customers get phone support with 1-hour resp (score: 0.08)


In [15]:
# =============================================================================
# RE-RANKING WITH A CROSS-ENCODER — second-pass "expert judge"
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHY RERANK?
# ---------------------------------------------------------------------------
# Hybrid retrieve is FAST but ROUGH: it scores query vs each chunk separately
# (bi-encoder / BM25), then returns a shortlist.
#
# Cross-encoder is SLOWER but SHARPER: it reads (query + chunk) TOGETHER and
# outputs one relevance score per pair. Use it only on the shortlist.
#
# Picture:
#   hybrid_retrieve → [Support, Security, Pricing, API, ...]  (top_k=5)
#         │
#         ▼  cross_encoder.predict( [(q, chunk1), (q, chunk2), ...] )
#   resorted by rerank_score → keep best 2 for the LLM prompt
#
# Terms:
#   Bi-encoder     — embed query and doc apart, compare vectors (FAISS path)
#   Cross-encoder  — one model forward on the pair (query, doc)
#   ms-marco-...   — checkpoint trained to rank passages for search queries
#   rerank_score   — higher = more relevant to THIS query
#

from sentence_transformers import CrossEncoder
from typing import List, Dict

# Downloads once; small MiniLM cross-encoder tuned on MS MARCO passage ranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, candidates: List[Dict], top_k: int = 2):
    """Score each (query, candidate text) pair; return top_k by rerank_score."""
    pairs = [(query, cand["text"]) for cand in candidates]
    scores = cross_encoder.predict(pairs)
    for i, cand in enumerate(candidates):
        cand["rerank_score"] = float(scores[i])
    candidates.sort(key=lambda x: x["rerank_score"], reverse=True)
    return candidates[:top_k]


# Example: hybrid shortlist, then cross-encoder refine
query = "What support do enterprise customers get?"
candidates = hybrid_retrieve(query, top_k=5)  # may only have 4 chunks total
print("Before rerank (hybrid order):")
for r in candidates:
    print(f"  {r['metadata']['title']}: hybrid={r['score']:.2f}")

reranked = rerank(query, candidates, top_k=2)
print("\nAfter rerank (cross-encoder):")
for r in reranked:
    print(f"  {r['metadata']['title']}: rerank_score={r['rerank_score']:.2f}")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# Before: titles ordered by hybrid blend (dense + BM25).
# After:  titles ordered by cross-encoder relevance to the support question.
#
# Expect Support near/at the top after rerank, e.g.:
#   Support: rerank_score=...   ← phone support, 1-hour SLA, Enterprise
#   ... maybe Pricing/Security with lower scores
#
# Numbers are model logits/scores — compare WITHIN one query; don't treat
# them as probabilities unless you calibrate.
#
# Sticky: retrieve broad → rerank narrow → send only the best chunks to the LLM.


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13929.72it/s]


Before rerank (hybrid order):
  Support: hybrid=0.73
  Pricing: hybrid=0.21
  Security: hybrid=0.06
  API Limits: hybrid=0.00

After rerank (cross-encoder):
  Support: rerank_score=8.43
  Pricing: rerank_score=5.91


In [18]:
# =============================================================================
# QUERY REWRITING — fix the question BEFORE you search
# =============================================================================
#
# Why rewrite?
#   Users ask casually: "what SOC stuff do we have?"
#   Docs say formally: "We are SOC 2 Type II compliant…"
#   Rewriting makes retrieval see a better string (or several strings).
#
# Picture:
#   user query ──► rewrite ──► hybrid_retrieve / FAISS
#                    │
#        ┌───────────┼───────────┐
#        ▼           ▼           ▼
#      HyDE     multi-query   decompose
#
# Methods in this cell:
#
#   HyDE (Hypothetical Document Embeddings)
#     Ask an LLM: "write a fake answer paragraph."
#     Embed/search THAT paragraph (often closer to real doc style than the
#     short question). Here we SIMULATE the LLM with a crafted string.
#
#   Multi-query
#     Expand into alternate phrasings / add keywords; retrieve with the
#     richer query (or retrieve per variant and merge — full version later).
#
#   Decompose
#     Split a complex question into sub-questions, retrieve for each,
#     then fuse (stubbed here).
#

def rewrite_query(query: str, method: str = "hyde") -> str:
    """Return a search-friendly rewrite of the user query (demo implementations)."""
    q = query.strip()

    if method == "hyde":
        # Simulate LLM "hypothetical document" — should SOUND like a playbook answer
        q_lower = q.lower()
        if "soc" in q_lower or "compliance" in q_lower or "security" in q_lower:
            return (
                "We are SOC 2 Type II compliant. Data is encrypted at rest and in "
                "transit. Enterprise customers can use their own encryption keys (BYOK)."
            )
        if "support" in q_lower or "sla" in q_lower:
            return (
                "Enterprise customers get phone support with 1-hour response time SLA. "
                "Standard support is 24/7 via chat and email."
            )
        if "price" in q_lower or "cost" in q_lower or "pricing" in q_lower:
            return (
                "Our Starter plan is $99/month, Professional is $299/month, "
                "and Enterprise is custom-priced."
            )
        # Generic fallback hypothetical
        return f"According to the sales playbook: {q}"

    if method == "multi_query":
        # Append helpful keywords when the query looks related
        alternatives = {
            "pricing": ["price", "cost", "how much", "fee", "subscription"],
            "security": ["soc", "compliance", "encrypt", "security", "byok"],
            "support": ["support", "sla", "phone", "response"],
        }
        extras = []
        for key, triggers in alternatives.items():
            if any(w in q.lower() for w in triggers):
                extras.append(key)
        if extras:
            return f"{q} " + " ".join(extras)
        return q

    if method == "decompose":
        # Stub: real version would return a LIST of sub-queries
        return q

    return q


# --- Demo: HyDE for a compliance question ---
original = "What SOC compliance do we have?"
rewritten = rewrite_query(original, "hyde")
print("Original:", original)
print("Rewritten (HyDE):", rewritten)

print("\nMulti-query variant:")
print(rewrite_query("how much for the pro plan?", "multi_query"))

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# Original:  What SOC compliance do we have?
# Rewritten (HyDE): We are SOC 2 Type II compliant. Data is encrypted...
#
#   The rewrite looks like a PLAYBOOK PARAGRAPH, not a question.
#   You would embed/search that paragraph so FAISS/BM25 find the Security
#   chunk more easily than the short informal question alone.
#
# Multi-query example:
#   "how much for the pro plan?" → "... pricing"
#   Extra keyword "pricing" helps BM25 hit the Pricing section title/text.
#
# Next step in a full pipeline:
#   q2 = rewrite_query(user_q, "hyde")
#   hits = hybrid_retrieve(q2) or retrieve(q2)
#   hits = rerank(user_q, hits)   # often rerank with the ORIGINAL user question


Original: What SOC compliance do we have?
Rewritten (HyDE): We are SOC 2 Type II compliant. Data is encrypted at rest and in transit. Enterprise customers can use their own encryption keys (BYOK).

Multi-query variant:
how much for the pro plan? pricing


In [20]:
# =============================================================================
# EVALUATION OF RETRIEVAL — prove the search quality (not just "feels good")
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHY EVALUATE?
# ---------------------------------------------------------------------------
# RAG fails in two common ways:
#   1) Wrong chunks retrieved  → LLM never saw the fact
#   2) Right chunks, bad answer → generation / prompt issue
# Classic IR metrics(Information Retrieval) below score (1). RAGAS-style metrics also score (2).
#
# Picture:
#   query → retrieve ranked list [docA, docB, docC, ...]
#   gold  → set of docs humans marked relevant
#   metrics compare ranked list vs gold
#
# ---------------------------------------------------------------------------
# CLASSIC IR METRICS (this cell implements these)
# ---------------------------------------------------------------------------
#   Precision@k
#     Of the top-k retrieved, what fraction are relevant?
#     "Among the 3 cards I handed the LLM, how many were useful?"
#
#   Recall@k
#     Of ALL relevant docs, what fraction appear in the top-k?
#     "Did we find the important pages at all?"
#
#   MRR = Mean Reciprocal Rank
#     1 / rank_of_first_relevant_doc  (0 if none found)
#     "How soon does the first good hit appear?" (1.0 = first place)
#
#   (Related, not coded here)
#   nDCG@k — graded relevance with position discount (common in search papers)
#   Hit@k  — 1 if any relevant in top-k, else 0 (simple success flag)
#   MAP    — Mean Average Precision across queries
#
# ---------------------------------------------------------------------------
# MARKET / FRAMEWORK EVAL TECHNIQUES (easy map)
# ---------------------------------------------------------------------------
# A) Retrieval-only (labels = which passages are relevant)
#      Precision/Recall/MRR/nDCG/Hit@k  ← industry IR staples
#      BEIR / MTEB-style benchmarks for embedding models
#
# B) End-to-end RAG (question → answer, with retrieved context)
#      Faithfulness / Groundedness
#        Does the answer stick to the retrieved text (not invent)?
#      Answer relevance
#        Does the answer actually address the question?
#      Context relevance / Context precision
#        Are the retrieved chunks useful for this question?
#      Context recall
#        Did retrieval cover what the gold answer needed?
#
# C) Popular toolkits
#      RAGAS (Retrieval-Augmented Generation Assessment Score) -
# LLM-as-judge + metrics like faithfulness, context precision
#      DeepEval  — pytest-like eval suite for LLM apps (RAG, agents, …)
#      TruLens / RAGChecker / ARES — tracing + RAG quality scoring
#      LangSmith / Phoenix — tracing runs + offline eval datasets
#
# Sticky for your earpiece:
#   Retrieval metrics → "did we find Pricing/Security?"
#   Faithfulness      → "did the whisper invent a fake price?"
#

from typing import List, Set


def precision_at_k(retrieved_docs: List, relevant_docs: Set, k: int) -> float:
    """Fraction of top-k retrieved items that are relevant."""
    if k <= 0:
        return 0.0
    retrieved_k = retrieved_docs[:k]
    hits = len(set(retrieved_k).intersection(set(relevant_docs)))
    return hits / k


def recall_at_k(retrieved_docs: List, relevant_docs: Set, k: int) -> float:
    """Fraction of all relevant items found inside the top-k."""
    if not relevant_docs:
        return 0.0
    retrieved_k = retrieved_docs[:k]
    hits = len(set(retrieved_k).intersection(set(relevant_docs)))
    return hits / len(relevant_docs)


def mrr(retrieved_docs: List, relevant_docs: Set) -> float:
    """Reciprocal rank of the first relevant hit (1, 1/2, 1/3, …)."""
    relevant_docs = set(relevant_docs)
    for i, doc in enumerate(retrieved_docs):
        if doc in relevant_docs:
            return 1.0 / (i + 1)
    return 0.0


# ---------------------------------------------------------------------------
# Tiny labeled eval on our playbook titles
# ---------------------------------------------------------------------------
# corpus order from earlier cells: 0 Pricing, 1 Security, 2 API Limits, 3 Support
# We evaluate by TITLE ids (stable labels).

title_by_query = {
    # query → set of relevant section titles
    "What is the price of Professional?": {"Pricing"},
    "Are we SOC 2 compliant?": {"Security"},
    "What support do enterprise customers get?": {"Support"},
}

print("Retrieval eval (titles as doc ids)\n")
for query, relevant in title_by_query.items():
    # Use hybrid if available, else dense retrieve
    if "hybrid_retrieve" in globals():
        hits = hybrid_retrieve(query, top_k=3)
    else:
        hits = retrieve(query, top_k=3)

    ranked_titles = [h["metadata"]["title"] for h in hits]
    p_at_1 = precision_at_k(ranked_titles, relevant, k=1)
    r_at_3 = recall_at_k(ranked_titles, relevant, k=3)
    mrr_v = mrr(ranked_titles, relevant)

    print(f"Q: {query}")
    print(f"  retrieved: {ranked_titles}")
    print(f"  relevant:  {sorted(relevant)}")
    print(f"  Precision@1={p_at_1:.2f}  Recall@3={r_at_3:.2f}  MRR={mrr_v:.2f}")
    print()

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# Example:
#   Q: What is the price of Professional?
#     retrieved: ['Pricing', 'Support', 'API Limits']
#     relevant:  ['Pricing']
#     Precision@1=1.00  Recall@3=1.00  MRR=1.00
#
#   Precision@1=1 → top hit was relevant (Pricing first). Great for RAG prompts.
#   Recall@3=1   → the relevant section appears somewhere in top-3.
#   MRR=1        → first relevant was rank 1; if it were rank 2 → MRR=0.5.
#
# If Precision@1 is often 0 but Recall@3 is 1:
#   retrieval finds the doc but not first → add reranking / query rewrite.
#
# Next level (market): build a small JSON eval set and run RAGAS faithfulness
# on (question, contexts, answer) triples after you wire generation.


Retrieval eval (titles as doc ids)

Q: What is the price of Professional?
  retrieved: ['API Limits', 'Pricing', 'Support']
  relevant:  ['Pricing']
  Precision@1=0.00  Recall@3=1.00  MRR=0.50

Q: Are we SOC 2 compliant?
  retrieved: ['Security', 'Support', 'API Limits']
  relevant:  ['Security']
  Precision@1=1.00  Recall@3=1.00  MRR=1.00

Q: What support do enterprise customers get?
  retrieved: ['Support', 'Pricing', 'Security']
  relevant:  ['Support']
  Precision@1=1.00  Recall@3=1.00  MRR=1.00



In [22]:
# =============================================================================
# INTEGRATE RAG AS AN ASSISTANT TOOL (MCP / function-calling style)
# =============================================================================
#
# Picture (same agent loop as week2, new tool):
#
#   User: "What's Enterprise support SLA?"
#      │
#      ▼
#   Model may output:  {"function_call": {"name": "search_playbook", "arguments": {"query": "..."}}}
#      │
#      ▼
#   YOUR code runs search_playbook(query)
#      │  rewrite (HyDE) → hybrid retrieve → rerank → format snippets
#      ▼
#   Tool result text fed back → model writes final earpiece line
#
# Sticky:
#   RAG engine = library lookup
#   Tool schema = menu entry the model can request
#   handle_tool_call = kitchen that cooks the order
#

from pathlib import Path
import sys

# Optional: register into week2 sales_tools registry if that package is on path
_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))


def search_playbook(query: str) -> str:
    """Full advanced-RAG pipeline → short grounded snippets for the LLM."""
    # 1) Rewrite for better retrieval (search with HyDE text)
    rewritten = rewrite_query(query, method="hyde")
    # 2) Hybrid shortlist
    candidates = hybrid_retrieve(rewritten, top_k=min(5, len(corpus)))
    # 3) Rerank with the ORIGINAL user question (what they actually asked)
    reranked = rerank(query, candidates, top_k=2)
    if not reranked:
        return "No relevant information found."

    # 4) Format citations the model can quote
    lines = ["Based on the sales playbook:"]
    for r in reranked:
        title = r["metadata"]["title"]
        lines.append(f"- [{title}] {r['text']}")
    return "\n".join(lines)


# JSON schema the model sees (function-calling / MCP tools/list entry)
SEARCH_PLAYBOOK_TOOL = {
    "name": "search_playbook",
    "description": (
        "Search the sales playbook for product details, pricing, security, support. "
        "Use when you need grounded product facts."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The question or topic to search for.",
            }
        },
        "required": ["query"],
    },
}

# Keep a local assistant tool list for this notebook
if "assistant_tools" not in globals():
    assistant_tools = []
# Avoid duplicate appends on re-run
assistant_tools = [t for t in assistant_tools if t.get("name") != "search_playbook"]
assistant_tools.append(SEARCH_PLAYBOOK_TOOL)

# Also merge into week2 `tools` registry when imported
try:
    from sales_tools import tools as _sales_tools, TOOL_IMPL as _TOOL_IMPL

    _sales_tools[:] = [t for t in _sales_tools if t.get("name") != "search_playbook"]
    _sales_tools.append(SEARCH_PLAYBOOK_TOOL)
    _TOOL_IMPL["search_playbook"] = search_playbook
    print("Registered search_playbook into sales_tools.TOOL_IMPL + tools schema list.")
except Exception as e:
    print(f"sales_tools not loaded ({e}); using assistant_tools only.")


def handle_tool_call(tool_name: str, arguments: dict) -> str:
    """Dispatch one tool call (DIY function-calling / MCP tools/call)."""
    if tool_name == "search_playbook":
        return search_playbook(arguments["query"])
    # Hook other tools here (get_product_price, …) or use TOOL_IMPL[name](**arguments)
    try:
        from sales_tools import TOOL_IMPL

        if tool_name in TOOL_IMPL:
            return TOOL_IMPL[tool_name](**arguments)
    except Exception:
        pass
    return f"Error: unknown tool '{tool_name}'"


# --- Demo (no LLM required) ---
demo_q = "What support do enterprise customers get?"
demo_out = handle_tool_call("search_playbook", {"query": demo_q})
print("\nTool schemas:", [t["name"] for t in assistant_tools])
print("\nsearch_playbook demo:")
print(demo_out)

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# Tool schemas: [..., 'search_playbook']
#   Model can now request this tool by name.
#
# search_playbook demo:
#   Based on the sales playbook:
#   - [Support] Standard support is 24/7...
#   (and maybe a second related section)
#
#   That STRING is what you'd append as "Function result: ..." in the
#   week2 assistant loop before asking the model for the final whisper.


sales_tools not loaded (No module named 'sales_tools'); using assistant_tools only.

Tool schemas: ['search_playbook']

search_playbook demo:
Based on the sales playbook:
- [Support] Standard support is 24/7 via chat and email. Enterprise customers get phone support with 1-hour response time SLA. Critical issues are escalated immediately.
- [Pricing] Our Starter plan is $99/month, Professional is $299/month, and Enterprise is custom-priced. All plans include 24/7 support. Enterprise customers receive a dedicated account manager.
